# KneeAxis — Step 1: Dataset Preparation & Validation

This notebook organizes, validates, and prepares manifests for the training datasets:
- **OAI (Osteoarthritis Initiative)**: Bilateral knee joint radiographs labeled with KL grades (0–4)
- **MOST Study**: Independent external validation radiographs
- **OAIZIB-CM (Zenodo 14934086)**: 3D Sagittal DESS MRI volumes with 6 anatomical classes

> **Data Integrity Rules:**
> - Splits are performed strictly at the **subject level** (no patient overlap between train/val/test).
> - No synthetic or guessed KL grades: manifests reflect only genuine dataset entries.

### 1. Initialize Paths & Configuration

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.common.paths import load_config, resolve_path
config = load_config()
print("Master configuration loaded successfully.")

### 2. Generate Dataset Manifests
Scans `datasets/` and creates reproducible CSV manifests in `manifests/`.

In [ ]:
from scripts.prepare_manifests import build_all_manifests
build_all_manifests(config)

### 3. Validate Dataset Integrity
Inspects file existence, image readability, volume dimensions, and Tr/Ts matching.

In [ ]:
from scripts.validate_datasets import validate_datasets
validation_report = validate_datasets(config)
for section, details in validation_report.items():
    print(f"{section}: {details}")

### 4. Inspect Manifest Distribution (Subject-Level Breakdown)

In [ ]:
import pandas as pd
oai_manifest = resolve_path(config['paths']['manifests'], project_root) / 'oai_kl_manifest.csv'
if oai_manifest.exists():
    df = pd.read_csv(oai_manifest)
    print(f"Total OAI Records:   {len(df)}")
    print(f"Unique Subjects:     {df['subject_id'].nunique()}")
    print("\nKL Grade Distribution:")
    print(df['kl_grade'].value_counts().sort_index())
else:
    print("No OAI manifest found yet. Download the OAI dataset to datasets/OAI/ first (see guides/DATASET_DOWNLOADS.md).")